# CDC Diabetes Prevention RAG

This notebook demonstrates the end-to-end retrieval-augmented generation (RAG) pipeline for the **CDC Diabetes Prevention RAG** project.

The production implementation lives in `implementation/`; this notebook imports those modules rather than duplicating the application logic.

**Pipeline**

`Question → BM25 + Dense Retrieval → Reciprocal Rank Fusion → CrossEncoder Reranking → CDC Evidence → Grounded LLM Answer`

The corpus consists of CDC *Preventing Chronic Disease* publications related to type 2 diabetes. The chatbot is intended for educational and public-health information and does not provide individualized diagnosis or medication advice.


## 1. Imports and project setup

The notebook is expected to be run from the repository root, where `implementation/` and `data/` are available.


In [1]:
import pandas as pd

from implementation.retrieval import BM25Retriever, load_chunks
from implementation.embeddings import DenseRetriever
from implementation.hybrid_retrieval import HybridRetriever
from implementation.reranker import Reranker
from implementation.prompts import build_user_prompt


c:\Users\gokif\projects\cdc_diabetes_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the CDC chunk corpus


In [2]:
chunks = load_chunks()

print(f"Chunks loaded: {len(chunks):,}")
print(f"Unique CDC documents: {len({chunk['cdc_id'] for chunk in chunks})}")


Chunks loaded: 3,154
Unique CDC documents: 74


In [3]:
pd.DataFrame(
    [
        {
            "chunk_id": chunk.get("chunk_id"),
            "cdc_id": chunk.get("cdc_id"),
            "title": chunk.get("title"),
            "page": chunk.get("page"),
            "screening_status": chunk.get("screening_status"),
        }
        for chunk in chunks[:10]
    ]
)

,chunk_id,cdc_id,title,page,screening_status
0,cdc_113215_p1_c1,113215,Type 2 Diabetes Among Filipino American Adults...,1,Include
1,cdc_113215_p1_c2,113215,Type 2 Diabetes Among Filipino American Adults...,1,Include
2,cdc_113215_p1_c3,113215,Type 2 Diabetes Among Filipino American Adults...,1,Include
3,cdc_113215_p1_c4,113215,Type 2 Diabetes Among Filipino American Adults...,1,Include
4,cdc_113215_p1_c5,113215,Type 2 Diabetes Among Filipino American Adults...,1,Include
5,cdc_113215_p2_c1,113215,Type 2 Diabetes Among Filipino American Adults...,2,Include
6,cdc_113215_p2_c2,113215,Type 2 Diabetes Among Filipino American Adults...,2,Include
7,cdc_113215_p2_c3,113215,Type 2 Diabetes Among Filipino American Adults...,2,Include
8,cdc_113215_p2_c4,113215,Type 2 Diabetes Among Filipino American Adults...,2,Include
9,cdc_113215_p2_c5,113215,Type 2 Diabetes Among Filipino American Adults...,2,Include


## 3. Example question


In [4]:
QUESTION = "How can lifestyle interventions prevent type 2 diabetes?"
QUESTION

'How can lifestyle interventions prevent type 2 diabetes?'

## 4. BM25 lexical retrieval

BM25 favors passages with strong keyword overlap with the query.


In [5]:
bm25 = BM25Retriever(chunks)
bm25_results = bm25.search(QUESTION, top_k = 5)

pd.DataFrame(
    [
        {
            "rank": item["rank"],
            "cdc_id": item.get("cdc_id"),
            "title": item.get("title"),
            "page": item.get("page"),
            "bm25_score": item.get("bm25_score"),
            "text": item.get("text", "")[:300],
        }
        for item in bm25_results
    ]
)

,rank,cdc_id,title,page,bm25_score,text
0,1,37180,A Randomized Controlled Trial Translating the ...,1,22.574692,with type 2 diabetes (2). Rising rates of pred...
1,2,22055,Diabetes Prevention in Hispanics: Report From ...,2,22.221689,use diabetes medical and self-management pract...
2,3,37180,A Randomized Controlled Trial Translating the ...,1,19.646379,in fasting glucose were great-\ner in the inte...
3,4,20499,Cost-Effectiveness Analysis of Efforts to Redu...,6,19.488632,et al. Reduction in the \nincidence of type 2 ...
4,5,150995,“Make Stories That Will Always Be There”: Eagl...,3,19.164329,"y, and social support.\n1.\nShare stories that..."


## 5. Dense semantic retrieval

Dense retrieval uses `sentence-transformers/multi-qa-mpnet-base-cos-v1`.

Generated embeddings are cached locally in `data/embeddings/`, so the full corpus does not need to be re-embedded after the cache is created.


In [6]:
dense = DenseRetriever(chunks)
dense_results = dense.search(QUESTION, top_k=5)

pd.DataFrame(
    [
        {
            "rank": item["rank"],
            "cdc_id": item.get("cdc_id"),
            "title": item.get("title"),
            "page": item.get("page"),
            "dense_score": item.get("dense_score"),
            "text": item.get("text", "")[:300],
        }
        for item in dense_results
    ]
)

Loading embedding model: sentence-transformers/multi-qa-mpnet-base-cos-v1


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 332.00it/s]


Loading cached embeddings:
C:\Users\gokif\projects\cdc_diabetes_rag\data\embeddings\cdc_embeddings.pt
Cached embeddings loaded: 3154


,rank,cdc_id,title,page,dense_score,text
0,1,20499,Cost-Effectiveness Analysis of Efforts to Redu...,1,0.841943,ials have demonstrated the \nefficacy of lifes...
1,2,37180,A Randomized Controlled Trial Translating the ...,1,0.818911,with type 2 diabetes (2). Rising rates of pred...
2,3,29611,A Comparison of Cardiometabolic Risk Factors i...,7,0.793821,The Indian Diabetes Prevention Programme\nshow...
3,4,151083,Prevalence of Testing for Diabetes Among US Ad...,4,0.793423,sociation between obesity and type 2 dia-\nbet...
4,5,36133,The State of Evaluation Research on Food Polic...,1,0.785483,condary prevention. Strong evidence\nsuggests ...


## 6. Hybrid retrieval with Reciprocal Rank Fusion

BM25 and dense retrieval use different score scales, so their rankings are combined with **Reciprocal Rank Fusion (RRF)** rather than by averaging raw scores.


In [7]:
hybrid = HybridRetriever(chunks)
hybrid_results = hybrid.search(
    query = QUESTION,
    top_k = 10,
    candidate_k = 30,
)

pd.DataFrame(
    [
        {
            "hybrid_rank": item["rank"],
            "cdc_id": item.get("cdc_id"),
            "title": item.get("title"),
            "page": item.get("page"),
            "bm25_rank": item.get("bm25_rank"),
            "dense_rank": item.get("dense_rank"),
            "rrf_score": item.get("rrf_score"),
        }
        for item in hybrid_results
    ]
)

Initializing BM25 retriever...
Initializing dense retriever...
Loading embedding model: sentence-transformers/multi-qa-mpnet-base-cos-v1


Loading weights: 100%|██████████| 199/199 [00:02<00:00, 98.41it/s]


Loading cached embeddings:
C:\Users\gokif\projects\cdc_diabetes_rag\data\embeddings\cdc_embeddings.pt
Cached embeddings loaded: 3154


,hybrid_rank,cdc_id,title,page,bm25_rank,dense_rank,rrf_score
0,1,37180,A Randomized Controlled Trial Translating the ...,1,1,2,0.032522
1,2,20499,Cost-Effectiveness Analysis of Efforts to Redu...,6,4,6,0.030777
2,3,20499,Cost-Effectiveness Analysis of Efforts to Redu...,1,11,1,0.030478
3,4,22055,Diabetes Prevention in Hispanics: Report From ...,2,2,11,0.030214
4,5,20102,Modifiable Risk Factors for Developing Diabete...,4,7,8,0.029631
5,6,20499,Cost-Effectiveness Analysis of Efforts to Redu...,6,9,7,0.029418
6,7,151083,Prevalence of Testing for Diabetes Among US Ad...,4,15,4,0.028958
7,8,20101,The Epidemic of Extreme Obesity Among American...,5,22,10,0.026481
8,9,20595,Forecasting Diabetes Prevalence in California:...,4,26,12,0.025517
9,10,50304,"Early Results of States’ Efforts to Support, S...",5,30,15,0.024444


## 7. CrossEncoder reranking

A CrossEncoder evaluates the query together with each hybrid candidate and reranks the candidate set.

Model: `cross-encoder/ms-marco-MiniLM-L6-v2`


In [8]:
reranker = Reranker()

candidates = hybrid.search(
    query = QUESTION,
    top_k = 20,
    candidate_k = 40,
)

evidence = reranker.rerank(
    query = QUESTION,
    candidates = candidates,
    top_k = 5,
)

pd.DataFrame(
    [
        {
            "final_rank": item["rank"],
            "cdc_id": item.get("cdc_id"),
            "title": item.get("title"),
            "page": item.get("page"),
            "reranker_score": item.get("reranker_score"),
            "source": item.get("landing_page"),
        }
        for item in evidence
    ]
)

Loading reranker model: cross-encoder/ms-marco-MiniLM-L6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 704.77it/s]


,final_rank,cdc_id,title,page,reranker_score,source
0,1,22055,Diabetes Prevention in Hispanics: Report From ...,2,0.999960,https://stacks.cdc.gov/view/cdc/22055
1,2,37180,A Randomized Controlled Trial Translating the ...,1,0.999866,https://stacks.cdc.gov/view/cdc/37180
2,3,20499,Cost-Effectiveness Analysis of Efforts to Redu...,1,0.999735,https://stacks.cdc.gov/view/cdc/20499
3,4,151083,Prevalence of Testing for Diabetes Among US Ad...,4,0.999442,https://stacks.cdc.gov/view/cdc/151083
4,5,20102,Modifiable Risk Factors for Developing Diabete...,2,0.999286,https://stacks.cdc.gov/view/cdc/20102


## 8. Inspect the final evidence passages


In [9]:
for index, item in enumerate(evidence, start = 1):
    print("=" * 100)
    print(f"[Source {index}]")
    print(f"Title: {item.get('title')}")
    print(f"CDC ID: {item.get('cdc_id')}")
    print(f"Page: {item.get('page')}")
    print(f"URL: {item.get('landing_page')}")
    print()
    print(item.get("text", "")[:1200])
    print()

[Source 1]
Title: Diabetes Prevention in Hispanics: Report From a Randomized Controlled Trial
CDC ID: 22055
Page: 2
URL: https://stacks.cdc.gov/view/cdc/22055

use diabetes medical and self-management practices, such as obtaining regular medical check-ups and self-monitoring 
of glucose levels, and are more likely to experience complications (9–11).
Lifestyle interventions can prevent type 2 diabetes in at-risk populations by improving glycemic control. The Diabetes 
Prevention Program (DPP), a large randomized controlled trial (RCT) of people at high risk for diabetes, demonstrated 
that a behavioral lifestyle intervention to lose weight and increase physical activity (PA) reduced development of type 2 
diabetes by 58% during a 3-year period (12). Although lifestyle interventions are more cost-effective than medications, 
reports identified difficulties in disseminating the original DPP intervention, including relatively high cost of one-on-
one delivery by behavioral experts, challen

## 9. Build the grounded medical prompt

The prompt requires the language model to use the supplied CDC evidence, cite source labels, preserve study/population limitations, and avoid individualized diagnosis or medication changes.


In [10]:
user_prompt = build_user_prompt(
    question = QUESTION,
    passages = evidence,
)

print(user_prompt[:6000])

Answer the question using only the CDC evidence below.

Question:
How can lifestyle interventions prevent type 2 diabetes?

CDC evidence:
[Source 1]
Title: Diabetes Prevention in Hispanics: Report From a Randomized Controlled Trial
CDC ID: 22055
Page: 2
URL: https://stacks.cdc.gov/view/cdc/22055

use diabetes medical and self-management practices, such as obtaining regular medical check-ups and self-monitoring 
of glucose levels, and are more likely to experience complications (9–11).
Lifestyle interventions can prevent type 2 diabetes in at-risk populations by improving glycemic control. The Diabetes 
Prevention Program (DPP), a large randomized controlled trial (RCT) of people at high risk for diabetes, demonstrated 
that a behavioral lifestyle intervention to lose weight and increase physical activity (PA) reduced development of type 2 
diabetes by 58% during a 3-year period (12). Although lifestyle interventions are more cost-effective than medications, 
reports identified difficul

## 10. End-to-end RAG answer

The production `CDCAnswerPipeline` combines retrieval, reranking, prompting, and the language-model call.

`OPENAI_API_KEY` must be available in the local `.env` file. The `.env` file should remain excluded from Git.


In [11]:
from implementation.answer import CDCAnswerPipeline

pipeline = CDCAnswerPipeline()
result = pipeline.answer(QUESTION)

print(result["answer"])

Loading CDC chunks...
Chunks loaded: 3154
Initializing hybrid retriever...
Initializing BM25 retriever...
Initializing dense retriever...
Loading embedding model: sentence-transformers/multi-qa-mpnet-base-cos-v1


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 162.54it/s]


Loading cached embeddings:
C:\Users\gokif\projects\cdc_diabetes_rag\data\embeddings\cdc_embeddings.pt
Cached embeddings loaded: 3154
Initializing reranker...
Loading reranker model: cross-encoder/ms-marco-MiniLM-L6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 981.62it/s]


Lifestyle interventions can prevent or delay type 2 diabetes by helping people at risk lose weight, improve dietary patterns, and increase physical activity. These changes can improve glycemic control and reduce the likelihood of developing diabetes. [Source 1] [Source 2]

Research from the Diabetes Prevention Program (DPP), a large randomized controlled trial involving people at high risk for diabetes, found that a behavioral lifestyle intervention focused on weight loss and increased physical activity reduced the development of type 2 diabetes by 58% over 3 years. [Source 1] A related CDC summary reports that the reduction in diabetes incidence remained at 10-year follow-up. [Source 2]

Evidence also supports lifestyle-change programs that focus on weight loss and physical activity, including the National Diabetes Prevention Program’s evidence-based lifestyle-change program. [Source 4] These findings apply primarily to people at increased risk, such as those with prediabetes, and do 

## 11. Retrieved evidence from the end-to-end pipeline


In [12]:
pd.DataFrame(
    [
        {
            "rank": index,
            "cdc_id": source.get("cdc_id"),
            "title": source.get("title"),
            "page": source.get("page"),
            "reranker_score": source.get("reranker_score"),
            "url": source.get("landing_page"),
        }
        for index, source in enumerate(result["sources"], start = 1)
    ]
)

,rank,cdc_id,title,page,reranker_score,url
0,1,22055,Diabetes Prevention in Hispanics: Report From ...,2,0.999960,https://stacks.cdc.gov/view/cdc/22055
1,2,37180,A Randomized Controlled Trial Translating the ...,1,0.999866,https://stacks.cdc.gov/view/cdc/37180
2,3,20499,Cost-Effectiveness Analysis of Efforts to Redu...,1,0.999735,https://stacks.cdc.gov/view/cdc/20499
3,4,151083,Prevalence of Testing for Diabetes Among US Ad...,4,0.999442,https://stacks.cdc.gov/view/cdc/151083
4,5,20102,Modifiable Risk Factors for Developing Diabete...,2,0.999286,https://stacks.cdc.gov/view/cdc/20102


## 12. Architecture summary

The repository separates production code from experimentation and evaluation:

- `implementation/retrieval.py` — BM25 lexical retrieval
- `implementation/embeddings.py` — dense semantic retrieval and persistent embedding cache
- `implementation/hybrid_retrieval.py` — Reciprocal Rank Fusion
- `implementation/reranker.py` — CrossEncoder reranking
- `implementation/prompts.py` — evidence-grounding and medical-safety instructions
- `implementation/answer.py` — end-to-end answer pipeline
- `app.py` — Gradio chatbot
- `evaluator.py` — retrieval evaluation runner
- `evaluation/retrieval_evaluation.ipynb` — retrieval analysis, graded relevance evaluation, and visualization
- `evaluation/generate_sample_answers.py` — benchmark answer generation
- `evaluation/evaluate_answers.py` — automated semantic comparison of generated and reference answers
- `scripts/map_reference_chunks.py` — mapping benchmark reference evidence to corpus chunks
- `scripts/validate_reference_chunks.py` — validation of benchmark reference mappings

The system was evaluated on a fixed 20-question benchmark.

Retrieval was evaluated using both known-reference metrics and pooled graded relevance judgments across BM25, dense, hybrid, and hybrid-plus-reranking methods. Dense retrieval achieved the strongest overall graded relevance performance, while hybrid reranking produced the highest Recall@5 by a negligible margin.

The final Hybrid + CrossEncoder RAG pipeline was also evaluated at the answer level using automated semantic comparison with reference answers. It achieved 39/40 points (97.5%), with 19 answers rated fully correct, 1 partially correct, and 0 incorrect.